**Import the libraries**

In [4]:
# ML tools
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [1]:
import pandas as pd

resume_df = pd.read_csv(r"C:\Users\bezis\Downloads\Smart-Resume-Job-Matching-System\data\preprocessed_resumes.csv")
job_df = pd.read_csv(r"C:\Users\bezis\Downloads\Smart-Resume-Job-Matching-System\data\preprocessed_jobs.csv")

print(resume_df.shape)
print(job_df.shape)


(9000, 5)
(1068, 9)


- Combine text for consistent vocabulary

In [2]:
# Combine text for consistent vocabulary
corpus = pd.concat([
    resume_df['preprocessed_text'],
    job_df['preprocessed_text']
])

**Create TF-IDF Vectorizer**

In [5]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),   # captures phrases like "machine learning"
    min_df=2,            # ignore rare words
    max_df=0.8           # ignore extremely common words
)

**Fit and Transform Text**

In [6]:
tfidf_matrix = tfidf.fit_transform(corpus)

In [7]:
resume_vectors = tfidf_matrix[:len(resume_df)]
job_vectors = tfidf_matrix[len(resume_df):]

**Calculate Similarity**

In [8]:
similarity_matrix = cosine_similarity(resume_vectors, job_vectors)

**Get Top Matching Jobs for a Resume**

In [9]:
def recommend_jobs(resume_index, top_n=5):

    scores = similarity_matrix[resume_index]

    top_jobs = scores.argsort()[-top_n:][::-1]

    results = job_df.iloc[top_jobs][['Title','ExperienceLevel','Skills']]
    results['match_score'] = scores[top_jobs]

    return results

In [10]:
recommend_jobs(100)

,Title,ExperienceLevel,Skills,match_score
567,Java Developer,Fresher,Spring Basics; Hibernate Basics; JUnit; TestNG,0.205444
561,Java Developer,Fresher,Spring Framework Basics; Hibernate Basics; Mav...,0.187233
568,Java Developer,Fresher,Core Java; OOP; Database Basics; Git; Team Col...,0.160382
569,Java Developer,Experienced,Advanced Java; Spring Boot; Hibernate; REST AP...,0.156361
559,Java Developer,Fresher,Core Java; OOP; Inheritance; Polymorphism; Enc...,0.150138


**Skill Matching Function**

In [11]:
def skill_match_score(resume_text, job_skills):
    resume_words = set(resume_text.split())
    job_skills = job_skills.lower().replace(';', ' ')
    job_words = set(job_skills.split())
    common = resume_words.intersection(job_words)
    return len(common)

**Hybrid Recommendation Function (TF-IDF + Skills + Experience)**

In [12]:
def recommend_jobs(resume_index, top_n=5, exp_filter=True):
    scores = similarity_matrix[resume_index]
    results = job_df.copy()
    results["similarity"] = scores

    resume_text = resume_df.iloc[resume_index]['preprocessed_text']
    resume_exp = resume_df.iloc[resume_index].get('ExperienceLevel', None)

    # Skill overlap bonus
    results["skill_score"] = results["Skills"].apply(lambda x: skill_match_score(resume_text, x))
    results["final_score"] = results["similarity"] + (results["skill_score"] * 0.02)

    # Optional: filter by same experience level
    if exp_filter and resume_exp in results.columns:
        results = results[results['ExperienceLevel'] == resume_exp]

    results = results.sort_values(by="final_score", ascending=False)
    return results[['Title','ExperienceLevel','Skills','final_score']].head(top_n)

**Resume Top Skills Extraction**

In [13]:
from collections import Counter  # make sure this is at the top

def extract_top_skills(resume_index, n=10):
    resume_text = resume_df.iloc[resume_index]['preprocessed_text']
    words = resume_text.split()
    counter = Counter(words)   # <-- use uppercase 'Counter'
    return counter.most_common(n)

In [14]:
# Top 5 jobs for the first resume
recommend_jobs(0)

# Top skills detected
extract_top_skills(0)

[('using', 40),
 ('application', 36),
 ('used', 32),
 ('sql', 27),
 ('java', 21),
 ('developed', 20),
 ('j', 19),
 ('server', 19),
 ('spring', 17),
 ('jsp', 16)]

In [15]:
recommend_jobs(0)



,Title,ExperienceLevel,Skills,final_score
560,Java Developer,Fresher,Java APIs; Concurrency Basics; JDBC; SQL (MySQ...,0.431523
0,.NET Developer,Fresher,C#; VB.NET basics; .NET Framework; .NET Core f...,0.360905
561,Java Developer,Fresher,Spring Framework Basics; Hibernate Basics; Mav...,0.349872
909,Software Tester (SDET),Experienced,Advanced Java; Python advanced; C# advanced; S...,0.338739
566,Java Developer,Fresher,Servlets; JSP; SQL Basics; JDBC; Git,0.337313
